# Quick Start: Running Simons on BOOM benchmark

This notebook is adapted from the [GiftEval repository](https://github.com/SalesforceAIResearch/gift-eval/tree/main/notebooks) and shows how to run [Simons](https://github.com/Abel-ai-lab/Simons) on the BOOM benchmark using the `SimonsPredictor` of the `simons` package, a gluonts-style predictor.

Simons is a family of four open time series foundation models from Abel AI Lab: [Simons-25M](https://huggingface.co/abel-lab/Simons-25M), [Simons-60M](https://huggingface.co/abel-lab/Simons-60M), [Simons-178M](https://huggingface.co/abel-lab/Simons-178M) and [Simons-297M](https://huggingface.co/abel-lab/Simons-297M). Key features:
- Zero-shot quantile forecasts at levels 0.1 to 0.9; the mean forecast is the mean of the nine quantiles
- Multivariate series are forecast jointly, in groups of at most 32 variates
- Max context length of 16,384 (25M, 60M) or 32,768 (178M, 297M)

Simons is evaluated zero-shot: its training data contains no BOOM data.

`CONTEXT_LENGTH` is `None` for all evaluations in this notebook: Simons reads the full history of each series, which is its released default. Set `CONTEXT_LENGTH = 2048` to cap the context as the other BOOM notebooks do.

To set up the environment, install Simons on top of the GiftEval environment described in the BOOM README:

```sh
pip install "simons @ git+https://github.com/Abel-ai-lab/Simons@ebc951aeec8971f7a7a89c0c83479aa4bf4a23ea"
```

Make sure you download the BOOM benchmark and set the `BOOM` environment variable correctly before running this notebook.

We will use the `Dataset` class from GiftEval to load the data and run the model.

## Download BOOM

`download_boom_benchmark` also sets the `BOOM` environment variable with the correct path, which is needed for running the evals below.

In [ ]:
import os
import json
from dotenv import load_dotenv
from dataset_utils import download_boom_benchmark

boom_path = "ChangeMe"
download_boom_benchmark(boom_path)
load_dotenv()

dataset_properties_map = json.load(open("./boom/boom_properties.json"))
all_datasets = list(dataset_properties_map.keys())
print(len(all_datasets))

In [ ]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
]

## Load Simons

Available sizes: `25M | 60M | 178M | 297M`. The model is loaded once and wrapped per dataset in a `SimonsPredictor`, which yields one gluonts `QuantileForecast` per test window with the keys `0.1` ... `0.9` and `mean`. All settings are the released defaults; on a GPU the transformer runs in bfloat16.

`SimonsBoomPredictor` only adds the optional context cap: history older than the last `CONTEXT_LENGTH` steps is set to NaN, which Simons reads as missing.

In [ ]:
import numpy as np
import torch
from simons import Simons
from simons.predictor import SimonsPredictor

SIZE = "297M"  # 25M | 60M | 178M | 297M
CHECKPOINT = f"abel-lab/Simons-{SIZE}"
CONTEXT_LENGTH = None  # None: the full history (released default) | 2048: as the other BOOM notebooks

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

simons_model = Simons.from_pretrained(CHECKPOINT, device=DEVICE)
print(f"Loaded {CHECKPOINT}: {sum(p.numel() for p in simons_model.parameters()):,} params")


class SimonsBoomPredictor(SimonsPredictor):
    def __init__(self, model, prediction_length, context_length=None, **kwargs):
        super().__init__(model, prediction_length=prediction_length, **kwargs)
        self.context_length = context_length

    def predict(self, dataset, **kwargs):
        if self.context_length is not None:
            dataset = map(self._crop, dataset)
        yield from super().predict(dataset, **kwargs)

    def _crop(self, entry):
        target = np.array(entry["target"], dtype=np.float32)
        target[..., : max(0, target.shape[-1] - self.context_length)] = np.nan
        return {**entry, "target": target}

## Evaluation

Iterate over all BOOM datasets and terms, rebuild a predictor per dataset (to pick up the dataset's `prediction_length`), and record metrics to `results/{model_name}/all_results.csv`.

Multivariate data stays multivariate. Several rolling windows can share a batch, but each window is a separate series for the model (variates only attend to variates of their own window), so no information leaks across windows of the same series.

In [ ]:
import csv

from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset

model_name = f"simons_{SIZE.lower()}"
output_dir = f"ChangeMe/{model_name}"
os.makedirs(output_dir, exist_ok=True)
csv_file_path = os.path.join(output_dir, "all_results.csv")

BATCH_SIZE = 1024  # lower this if you OOM on big multivariate windows

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
            "dataset_size",
        ]
    )

for ds_num, ds_name in enumerate(all_datasets):
    print(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")
    dataset_term = dataset_properties_map[ds_name]["term"]
    for term in ["short", "medium", "long"]:
        if (term == "medium" or term == "long") and dataset_term == "short":
            continue

        ds_freq = dataset_properties_map[ds_name]["frequency"]
        ds_config = f"{ds_name}/{ds_freq}/{term}"

        # Keep multivariate data multivariate so Simons forecasts the variates jointly
        dataset = Dataset(name=ds_name, term=term, to_univariate=False, storage_env_var="BOOM")
        season_length = get_seasonality(dataset.freq)
        dataset_size = len(dataset.test_data)
        print(f"  term={term} target_dim={dataset.target_dim} pred_len={dataset.prediction_length} size={dataset_size}")

        predictor = SimonsBoomPredictor(
            simons_model,
            prediction_length=dataset.prediction_length,
            context_length=CONTEXT_LENGTH,
            batch_size=BATCH_SIZE,
        )

        res = evaluate_model(
            predictor,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=1024,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )

        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(
                [
                    ds_config,
                    model_name,
                    res["MSE[mean]"].iloc[0],
                    res["MSE[0.5]"].iloc[0],
                    res["MAE[0.5]"].iloc[0],
                    res["MASE[0.5]"].iloc[0],
                    res["MAPE[0.5]"].iloc[0],
                    res["sMAPE[0.5]"].iloc[0],
                    res["MSIS"].iloc[0],
                    res["RMSE[mean]"].iloc[0],
                    res["NRMSE[mean]"].iloc[0],
                    res["ND[0.5]"].iloc[0],
                    res["mean_weighted_sum_quantile_loss"].iloc[0],
                    dataset_properties_map[ds_name]["domain"],
                    dataset_properties_map[ds_name]["num_variates"],
                    dataset_size,
                ]
            )
        print(f"  wrote row for {ds_config}")

## Results

Running the above cell will generate `all_results.csv` under `results/simons_<size>/`. We can display it:

In [ ]:
import pandas as pd
df = pd.read_csv(output_dir + "/all_results.csv")
df